# Gold Layer Data Monitoring
Validate row counts, schema consistency, join integrity, and dropped rows between Silver → Gold transformations.

In [ ]:
%%sql
USE DATABASE LEAGUE_RECORDS;

USE SCHEMA GOLD;

## 1. Object Inventory
Take stocks of all items under the Gold Layer.
1. `MATCHEND_PLAYER_STATS`: Match-player end game snapshot.
2. `CHAMPION_INTERVALS`: Champion aggregated performance over time.
3. `CHAMPION_OVERVIEWS`: Champion aggregated info e.g. win-rates, ban-rates, etc.
4. `MATCHEND_PIVOT_TEAMSTATS`: Match-grain pivot views.
5. `ITEM_RECOMMENDATIONS`: Champion and item usage rate, recommendations, etc.
6. `DIFF_INTERVALS`: Match-player-minute economy diff snapshot.


In [ ]:
%%sql
SHOW DYNAMIC TABLES IN SCHEMA GOLD;

## 2. Refresh History
Query the gold layer refreshing from silver.
> **Note**: Because this is a simulated pipeline, all the gold dynamic table will not refresh unless there's actually new data so they will all be 'Behind schedule'

In [ ]:
%%sql
SELECT
    "name" AS DT_NAME,
    "scheduling_state" AS STATE,
    "target_lag" AS TARGET_LAG,
    "data_timestamp" AS LAST_REFRESH_AT,
    DATEDIFF('minute', "data_timestamp", CURRENT_TIMESTAMP()) AS ACTUAL_LAG_MINUTES,
    CASE
        WHEN "scheduling_state" != 'ACTIVE' THEN 'NOT ACTIVE'
        WHEN DATEDIFF('minute', "data_timestamp", CURRENT_TIMESTAMP()) >
             REGEXP_SUBSTR("target_lag", '\\d+')::INT * 2 THEN 'BEHIND SCHEDULE'
        ELSE 'ON TRACK'
    END AS FRESHNESS_STATUS
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));

In [ ]:
%%sql
SELECT
    NAME AS DT_NAME,
    STATE AS REFRESH_STATE,
    REFRESH_START_TIME,
    REFRESH_END_TIME,
    DATEDIFF('second', REFRESH_START_TIME, REFRESH_END_TIME) AS REFRESH_DURATION_SEC,
    STATISTICS:"numInsertedRows"::INT AS ROWS_INSERTED,
    STATISTICS:"numDeletedRows"::INT AS ROWS_DELETED,
    STATISTICS:"numCopiedRows"::INT AS ROWS_COPIED
FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(
    NAME_PREFIX => 'LEAGUE_RECORDS.GOLD.'
))
ORDER BY REFRESH_END_TIME DESC
LIMIT 30;

## 2. Importing from Silver 
Validate that gold grain matches expectations:
- `MATCHEND_PLAYER_STATS` should have 1 row per (MATCH_ID, PARTICIPANT_POS_ID) = same distinct combos as SILVER.PLAYERS
- `MATCHEND_PIVOT_TEAMSTATS` should have 1 row per MATCH_ID = same count as SILVER.MATCHES
- `CHAMPION_OVERVIEWS` row count ≈ distinct champions in SILVER.CHAMPIONS_REF (minus ID=0)

Identify any dropped records between `SILVER` and `GOLD`.


In [ ]:
%%sql
WITH CHECKS AS (
    SELECT
        'MATCHEND_PLAYER_STATS' AS CHECK_NAME,
        (SELECT COUNT(*) FROM GOLD.MATCHEND_PLAYER_STATS) AS GOLD_ROWS,
        (SELECT COUNT(DISTINCT MATCH_ID || '|' || PARTICIPANT_POS_ID) FROM SILVER.PLAYERS) AS EXPECTED_ROWS
    UNION ALL
    SELECT
        'MATCHEND_PIVOT_TEAMSTATS',
        (SELECT COUNT(*) FROM GOLD.MATCHEND_PIVOT_TEAMSTATS),
        (SELECT COUNT(DISTINCT MATCH_ID) FROM SILVER.INTERVALS)
    UNION ALL
    SELECT
        'CHAMPION_OVERVIEWS',
        (SELECT COUNT(*) FROM GOLD.CHAMPION_OVERVIEWS),
        (SELECT COUNT(*) FROM SILVER.CHAMPIONS_REF WHERE CHAMPION_ID != 0)
    UNION ALL
    SELECT 
        'DIFF_INTERVALS',
        (SELECT COUNT(*) FROM GOLD.DIFF_INTERVALS),
        (SELECT COUNT(*) FROM SILVER.INTERVALS)
)

SELECT
    CHECK_NAME,
    GOLD_ROWS,
    EXPECTED_ROWS,
    GOLD_ROWS - EXPECTED_ROWS AS ROW_DIFF,
    CASE
        WHEN GOLD_ROWS = EXPECTED_ROWS THEN 'PASS'
        WHEN GOLD_ROWS < EXPECTED_ROWS THEN 'ROWS DROPPED'
        ELSE 'MORE ROWS THAN EXPECTED'
    END AS STATUS
FROM CHECKS;


In [ ]:
%%sql
WITH SILVER_KEYS AS (
    SELECT DISTINCT MATCH_ID, PARTICIPANT_POS_ID
    FROM SILVER.PLAYERS
),

GOLD_KEYS AS (
    SELECT DISTINCT MATCH_ID, PARTICIPANT_POS_ID
    FROM GOLD.MATCHEND_PLAYER_STATS
),

MISSING AS (
    SELECT S.MATCH_ID, S.PARTICIPANT_POS_ID
    FROM SILVER_KEYS AS S
    LEFT JOIN GOLD_KEYS AS G
        ON S.MATCH_ID = G.MATCH_ID 
        AND S.PARTICIPANT_POS_ID = G.PARTICIPANT_POS_ID
    WHERE G.MATCH_ID IS NULL
)

SELECT
    COUNT(*) AS ROWS_DROPPED,
    (SELECT COUNT(*) FROM SILVER_KEYS) AS TOTAL_SILVER_ROWS,
    ROUND(
        COUNT(*) / (SELECT COUNT(*) FROM SILVER_KEYS) * 100
    , 2) AS PCT_DROPPED
FROM MISSING;


In [ ]:
%%sql
WITH SILVER_MATCHES AS (
    SELECT DISTINCT MATCH_ID 
    FROM SILVER.MATCHES
),

GOLD_MATCHES AS (
    SELECT DISTINCT MATCH_ID 
    FROM GOLD.MATCHEND_PIVOT_TEAMSTATS
),

MISSING AS (
    SELECT S.MATCH_ID
    FROM SILVER_MATCHES AS S
    LEFT JOIN GOLD_MATCHES AS G 
        ON S.MATCH_ID = G.MATCH_ID
    WHERE G.MATCH_ID IS NULL
)

SELECT
    COUNT(*) AS MATCHES_DROPPED,
    (SELECT COUNT(*) FROM SILVER_MATCHES) AS TOTAL_SILVER_MATCHES,
    ROUND(
        COUNT(*) / (SELECT COUNT(*) FROM SILVER_MATCHES) * 100
    , 2) AS PCT_DROPPED
FROM MISSING;


Per table accuracy, statistics, null checks.

## 3. Sample Records: Gold vs Silver
Pull a few records and compare between silver and gold they carry the same informations.

In [ ]:
%%sql
WITH SAMPLE_PLAYERS AS (
    SELECT MATCH_ID, PARTICIPANT_POS_ID
    FROM GOLD.MATCHEND_PLAYER_STATS
    ORDER BY RANDOM()
    LIMIT 30
),

SILVER_LAST_INTERVAL AS (
    SELECT *
    FROM SILVER.INTERVALS
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY MATCH_ID, PARTICIPANT_POS_ID
        ORDER BY MINUTE DESC
    ) = 1
)

SELECT
    -- Primary key
    G.MATCH_ID AS SAMPLED_MATCH_ID,
    G.PARTICIPANT_POS_ID AS SAMPLED_PARTICIPANT_POS_ID,
    -- The sampled player's records in GOLD.MATCHEND_PLAYER_STATS
    G.KILLS AS GOLD_KILLS,
    G.DEATHS AS GOLD_DEATHS,
    G.ASSISTS AS GOLD_ASSISTS,
    G.TOTAL_GOLD AS GOLD_TOTAL_GOLD,
    -- The sampled player's records in SILVER.INTERVALS
    PIV.KILLS AS SILVER_LAST_KILLS,
    PIV.DEATHS AS SILVER_LAST_DEATHS,
    PIV.ASSISTS AS SILVER_LAST_ASSISTS,
    PIV.TOTAL_GOLD AS SILVER_LAST_GOLD,
    PIV.MINUTE AS SILVER_LAST_MINUTE,
    -- Verify joined from Gold is same as silver
    CASE 
        WHEN G.KILLS = PIV.KILLS 
            AND G.DEATHS = PIV.DEATHS
            AND G.ASSISTS = PIV.ASSISTS 
            AND G.TOTAL_GOLD = PIV.TOTAL_GOLD
            THEN 'MATCH'
        ELSE 'MISMATCH' 
    END AS VERIFICATION
FROM GOLD.MATCHEND_PLAYER_STATS AS G
JOIN SAMPLE_PLAYERS AS SP
    ON G.MATCH_ID = SP.MATCH_ID 
    AND G.PARTICIPANT_POS_ID = SP.PARTICIPANT_POS_ID
JOIN SILVER_LAST_INTERVAL AS PIV
    ON PIV.MATCH_ID = SP.MATCH_ID
    AND PIV.PARTICIPANT_POS_ID = SP.PARTICIPANT_POS_ID;


In [ ]:
%%sql
WITH SAMPLE_MATCHES AS (
    SELECT MATCH_ID
    FROM GOLD.MATCHEND_PIVOT_TEAMSTATS
    ORDER BY RANDOM()
    LIMIT 30
),

SILVER_LAST_INTERVAL AS (
    SELECT
        MATCH_ID,
        TEAM,
        TEAM_KILLS,
        TEAM_TOWERS,
        TEAM_INHIBITORS,
        TEAM_DRAGONS,
        TEAM_BARONS,
        TEAM_HERALDS,
        TEAM_VOID_GRUBS
    FROM SILVER.INTERVALS
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY MATCH_ID, TEAM
        ORDER BY MINUTE DESC
    ) = 1
)

SELECT
    G.MATCH_ID AS SAMPLED_MATCH_ID,
    -- Gold blue vs Silver blue
    G.BLUE_KILLS AS GOLD_BLUE_KILLS,
    BLUE.TEAM_KILLS AS SILVER_BLUE_KILLS,
    G.BLUE_TOWERS AS GOLD_BLUE_TOWERS,
    BLUE.TEAM_TOWERS AS SILVER_BLUE_TOWERS,
    G.BLUE_INHIBITORS AS GOLD_BLUE_INHIBITORS,
    BLUE.TEAM_INHIBITORS AS SILVER_BLUE_INHIBITORS,
    -- Gold red vs Silver red
    G.RED_KILLS AS GOLD_RED_KILLS,
    RED.TEAM_KILLS AS SILVER_RED_KILLS,
    G.RED_DRAGONS AS GOLD_RED_DRAGONS,
    RED.TEAM_DRAGONS AS SILVER_RED_DRAGONS,
    G.RED_INHIBITORS AS GOLD_RED_INHIBITORS,
    RED.TEAM_INHIBITORS AS SILVER_RED_INHIBITORS,
    -- Verification
    CASE
        WHEN G.BLUE_KILLS = BLUE.TEAM_KILLS
            AND G.RED_KILLS = RED.TEAM_KILLS
            AND G.BLUE_TOWERS = BLUE.TEAM_TOWERS
            AND G.RED_TOWERS = RED.TEAM_TOWERS
            AND G.BLUE_INHIBITORS = BLUE.TEAM_INHIBITORS
            AND G.RED_INHIBITORS = RED.TEAM_INHIBITORS
            AND G.BLUE_DRAGONS = BLUE.TEAM_DRAGONS
            AND G.RED_DRAGONS = RED.TEAM_DRAGONS
            AND G.BLUE_BARONS = BLUE.TEAM_BARONS
            AND G.RED_BARONS = RED.TEAM_BARONS
            THEN 'MATCH'
        ELSE 'MISMATCH'
    END AS VERIFICATION
FROM GOLD.MATCHEND_PIVOT_TEAMSTATS AS G
JOIN SAMPLE_MATCHES AS SM ON G.MATCH_ID = SM.MATCH_ID
JOIN SILVER_LAST_INTERVAL AS BLUE
    ON BLUE.MATCH_ID = G.MATCH_ID 
    AND BLUE.TEAM = 'BLUE'
JOIN SILVER_LAST_INTERVAL AS RED
    ON RED.MATCH_ID = G.MATCH_ID 
    AND RED.TEAM = 'RED';
